In [ ]:
import pandas as pd

# Load cleaned trade data
trade_df = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/cleaned/cleaned_trade_flows.csv")

# ---------------------------------------
# Extract countries from exporter/importer
# ---------------------------------------

exporters = trade_df[['exporter']].copy()
exporters.columns = ['country']

importers = trade_df[['importer']].copy()
importers.columns = ['country']

# Combine both
trade_geo = pd.concat([exporters, importers])

# Remove duplicates
trade_geo = trade_geo.drop_duplicates()

# Reset index
trade_geo = trade_geo.reset_index(drop=True)

# Create surrogate key
trade_geo['trade_geo_sk'] = range(1, len(trade_geo) + 1)

# Reorder columns
trade_geo = trade_geo[
    ['trade_geo_sk', 'country']
]

# Save dimension
trade_geo.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_trade_geography.csv",
    index=False
)

print(trade_geo.head(5))

In [ ]:
import pandas as pd

# ---------------------------------------
# Load files
# ---------------------------------------

trade_df = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/cleaned/cleaned_trade_flows.csv")

geo_df = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_trade_geography.csv")

date_df = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/dimensions/dim_date.csv")
# ---------------------------------------
# EXPORTER JOIN
# ---------------------------------------

geo_exporter = geo_df.rename(columns={
    'trade_geo_sk': 'exporter_geo_sk',
    'country': 'exporter'
})

trade_df = trade_df.merge(
    geo_exporter,
    on='exporter',
    how='left'
)

# ---------------------------------------
# IMPORTER JOIN
# ---------------------------------------

geo_importer = geo_df.rename(columns={
    'trade_geo_sk': 'importer_geo_sk',
    'country': 'importer'
})

trade_df = trade_df.merge(
    geo_importer,
    on='importer',
    how='left'
)

# ---------------------------------------
# DATE JOIN
# ---------------------------------------

date_lookup = (
    date_df
    .sort_values('date')
    .groupby('year')
    .first()
    .reset_index()[['year', 'date_key']]
)

trade_df = trade_df.merge(
    date_lookup,
    on='year',
    how='left'
)

# ---------------------------------------
# FINAL FACT
# ---------------------------------------

fact_trade = trade_df[[
    'date_key',
    'exporter_geo_sk',
    'importer_geo_sk',
    'trade_category',
    'key_goods',
    'trade_value_bn_usd',
    'yoy_growth_pct',
    'effective_tariff_rate_pct',
    'trade_active',
    'supply_chain_integrated',
    'concentration_risk'
]]

# ---------------------------------------
# SAVE FACT READY FILE
# ---------------------------------------

fact_trade.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/curated_trade_fact_ready.csv",
    index=False
)

print(fact_trade.head())